# NB09b — Ben-Sarc FGM + AWP final adversarial attempt

This notebook is intentionally isolated from the thesis master outputs.

It trains only on **Ben-Sarc binary** using:

- `csebuetnlp/banglabert` with its model-native tokenizer
- full fine-tuning
- a stronger classifier head using `[CLS] + mean pooling`
- weighted label-smoothed cross-entropy
- AdamW with layer-wise learning-rate decay and cosine schedule
- FGM with epsilon `0.5`
- AWP with alpha `0.01`
- early stopping and best-checkpoint selection by **validation loss**

All outputs are saved under:

`04_outputs/test_awp/`

Nothing from the main thesis result folders is overwritten by this notebook.

In [1]:
import importlib, subprocess, sys

def ensure(pip_name, import_name=None):
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", pip_name], check=False)

for pkg, imp in [
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("scikit-learn", "sklearn"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
]:
    ensure(pkg, imp)

print("Dependencies checked")

Dependencies checked


In [2]:
import os, json, time, random, shutil, warnings, unicodedata, re, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

import transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

HAS_CUDA = torch.cuda.is_available()
HAS_BF16 = HAS_CUDA and torch.cuda.is_bf16_supported()
DEVICE = "cuda" if HAS_CUDA else "cpu"

if HAS_CUDA:
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", HAS_BF16)
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
else:
    print("CUDA not available; this notebook is intended for RunPod/A40 GPU training.")

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if HAS_CUDA:
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

torch: 2.8.0+cu128
transformers: 4.40.0
GPU: NVIDIA A40
BF16 supported: True


In [3]:
def find_repo_root():
    candidates = [Path("/workspace/Sarcasm_detection"), Path.cwd()] + list(Path.cwd().resolve().parents)
    for c in candidates:
        if (c / "01_data").exists():
            return c.resolve()
    raise RuntimeError("Could not locate repo root. Run from inside the thesis repo after NB01 has created 01_data/.")

ROOT = find_repo_root()
SPLITS = ROOT / "01_data" / "interim" / "splits"

OUTROOT = ROOT / "04_outputs" / "test_awp"
TABLES = OUTROOT / "tables"
FIGS = OUTROOT / "figures"
PRED = OUTROOT / "predictions"
CKPT = OUTROOT / "checkpoints"
CONFIGS = OUTROOT / "configs"
LOGS = OUTROOT / "logs"

for p in [OUTROOT, TABLES, FIGS, PRED, CKPT, CONFIGS, LOGS]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT   :", ROOT)
print("SPLITS :", SPLITS, "| exists:", SPLITS.exists())
print("OUTROOT:", OUTROOT)
assert SPLITS.exists(), "Splits folder missing. Run NB01 first."

ROOT   : /workspace/Sarcasm_detection
SPLITS : /workspace/Sarcasm_detection/01_data/interim/splits | exists: True
OUTROOT: /workspace/Sarcasm_detection/04_outputs/test_awp


In [4]:
TASK = "ben_sarc_binary"
LABEL_COL = "label_binary"
NUM_LABELS = 2

MODEL_NAME = "csebuetnlp/banglabert"
RUN_NAME = "09b_fgm_awp_best_head"

MAX_LENGTH = 192
EPOCHS = 8
PATIENCE = 2
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
GRAD_ACCUM_STEPS = 1

BASE_LR = 2e-5
HEAD_LR = 1e-4
LLRD_DECAY = 0.95
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.08
LR_SCHEDULER = "cosine"
MAX_GRAD_NORM = 1.0

HIDDEN_DROPOUT = 0.20
ATTENTION_DROPOUT = 0.20
HEAD_DROPOUT = 0.30
LABEL_SMOOTHING = 0.05

FGM_EPSILON = 0.5
AWP_ALPHA = 0.01
AWP_EPS = 1e-3
AWP_START_EPOCH = 1.0
FGM_LOSS_WEIGHT = 1.0
AWP_LOSS_WEIGHT = 1.0

# With adversarial extra backward passes, bf16 is preferred on A40. fp16 is kept off for stability.
USE_BF16 = bool(HAS_BF16)
USE_FP16 = False
GRADIENT_CHECKPOINTING = False

config = {
    "run_name": RUN_NAME,
    "task": TASK,
    "model_name": MODEL_NAME,
    "tokenizer": "model-native AutoTokenizer from csebuetnlp/banglabert",
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "batch_size": BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "base_lr": BASE_LR,
    "head_lr": HEAD_LR,
    "llrd_decay": LLRD_DECAY,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "lr_scheduler": LR_SCHEDULER,
    "hidden_dropout": HIDDEN_DROPOUT,
    "attention_dropout": ATTENTION_DROPOUT,
    "head_dropout": HEAD_DROPOUT,
    "label_smoothing": LABEL_SMOOTHING,
    "fgm_epsilon": FGM_EPSILON,
    "awp_alpha": AWP_ALPHA,
    "awp_eps": AWP_EPS,
    "awp_start_epoch": AWP_START_EPOCH,
    "fgm_loss_weight": FGM_LOSS_WEIGHT,
    "awp_loss_weight": AWP_LOSS_WEIGHT,
    "early_stop_metric": "eval_loss",
    "optimizer": "AdamW + layer-wise LR decay + cosine schedule",
    "loss_function": "class-weighted label-smoothed cross entropy",
    "use_bf16": USE_BF16,
    "use_fp16": USE_FP16,
}

json.dump(config, open(CONFIGS / "09b_fgm_awp_config.json", "w"), indent=2)
print(json.dumps(config, indent=2))

{
  "run_name": "09b_fgm_awp_best_head",
  "task": "ben_sarc_binary",
  "model_name": "csebuetnlp/banglabert",
  "tokenizer": "model-native AutoTokenizer from csebuetnlp/banglabert",
  "max_length": 192,
  "epochs": 8,
  "patience": 2,
  "batch_size": 32,
  "eval_batch_size": 64,
  "gradient_accumulation_steps": 1,
  "base_lr": 2e-05,
  "head_lr": 0.0001,
  "llrd_decay": 0.95,
  "weight_decay": 0.01,
  "warmup_ratio": 0.08,
  "lr_scheduler": "cosine",
  "hidden_dropout": 0.2,
  "attention_dropout": 0.2,
  "head_dropout": 0.3,
  "label_smoothing": 0.05,
  "fgm_epsilon": 0.5,
  "awp_alpha": 0.01,
  "awp_eps": 0.001,
  "awp_start_epoch": 1.0,
  "fgm_loss_weight": 1.0,
  "awp_loss_weight": 1.0,
  "early_stop_metric": "eval_loss",
  "optimizer": "AdamW + layer-wise LR decay + cosine schedule",
  "loss_function": "class-weighted label-smoothed cross entropy",
  "use_bf16": true,
  "use_fp16": false
}


In [5]:
_ZW = dict.fromkeys(map(ord, ["\u200b", "\u200c", "\u200d", "\ufeff"]), None)

def norm_key(s):
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFC", s).translate(_ZW)
    s = re.sub(r"\s+", " ", s).strip()
    return s.casefold()

def assert_no_leak(tr, va, te, col="text"):
    a = {norm_key(x) for x in tr[col]}
    b = {norm_key(x) for x in va[col]}
    c = {norm_key(x) for x in te[col]}
    overlaps = {"train_val": len(a & b), "train_test": len(a & c), "val_test": len(b & c)}
    assert overlaps["train_val"] == 0, overlaps
    assert overlaps["train_test"] == 0, overlaps
    assert overlaps["val_test"] == 0, overlaps
    return overlaps

def load_ben_sarc():
    def rd(split):
        f = SPLITS / f"{TASK}_{split}.csv"
        df = pd.read_csv(f)
        assert "text" in df.columns, f"{f} missing text column"
        assert LABEL_COL in df.columns, f"{f} missing {LABEL_COL} column"
        df = df[["text", LABEL_COL]].dropna(subset=["text"]).copy()
        df["text"] = df["text"].astype(str)
        df[LABEL_COL] = df[LABEL_COL].astype(int)
        return df
    tr, va, te = rd("train"), rd("val"), rd("test")
    leak = assert_no_leak(tr, va, te)
    return tr, va, te, leak

train_df, val_df, test_df, leak = load_ben_sarc()
print("Leak check:", leak)
print("Sizes: train / val / test =", len(train_df), len(val_df), len(test_df))
print("Train label counts:")
print(train_df[LABEL_COL].value_counts().sort_index())

counts = train_df[LABEL_COL].value_counts().sort_index().reindex(range(NUM_LABELS), fill_value=0).values.astype(np.float32)
class_weights_np = len(train_df) / (NUM_LABELS * np.maximum(counts, 1.0))
class_weights = torch.tensor(class_weights_np, dtype=torch.float32)
print("Class weights:", class_weights.tolist())

Leak check: {'train_val': 0, 'train_test': 0, 'val_test': 0}
Sizes: train / val / test = 20498 2562 2563
Train label counts:
label_binary
0    10249
1    10249
Name: count, dtype: int64
Class weights: [1.0, 1.0]


In [6]:
def load_tokenizer(model_name):
    try:
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    except Exception as e:
        print("Fast tokenizer unavailable; falling back to slow tokenizer:", repr(e))
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    return tok

tokenizer = load_tokenizer(MODEL_NAME)
print("Tokenizer class:", type(tokenizer).__name__)
print("Vocab size:", getattr(tokenizer, "vocab_size", None))

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = torch.tensor(list(labels), dtype=torch.long)
        self.enc = tokenizer(
            self.texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item["labels"] = self.labels[idx]
        return item

train_ds = TextDataset(train_df["text"], train_df[LABEL_COL], tokenizer, MAX_LENGTH)
val_ds = TextDataset(val_df["text"], val_df[LABEL_COL], tokenizer, MAX_LENGTH)
test_ds = TextDataset(test_df["text"], test_df[LABEL_COL], tokenizer, MAX_LENGTH)
print("Datasets ready")

Tokenizer class: ElectraTokenizerFast
Vocab size: 32000
Datasets ready


In [7]:
class WeightedLabelSmoothingCE(nn.Module):
    def __init__(self, weight=None, smoothing=0.05):
        super().__init__()
        if weight is not None:
            self.register_buffer("weight", weight.float())
        else:
            self.weight = None
        self.smoothing = float(smoothing)

    def forward(self, logits, target):
        log_probs = F.log_softmax(logits, dim=-1)
        nll = -log_probs.gather(dim=-1, index=target.unsqueeze(1)).squeeze(1)
        smooth = -log_probs.mean(dim=-1)
        loss = (1.0 - self.smoothing) * nll + self.smoothing * smooth
        if self.weight is not None:
            w = self.weight.to(logits.device)[target]
            loss = loss * (w / w.mean().clamp_min(1e-8))
        return loss.mean()

class BanglaBertBestHead(nn.Module):
    def __init__(self, model_name, num_labels=2, class_weights=None, label_smoothing=0.05,
                 hidden_dropout=0.20, attention_dropout=0.20, head_dropout=0.30,
                 gradient_checkpointing=False):
        super().__init__()
        self.num_labels = num_labels
        cfg = AutoConfig.from_pretrained(model_name)
        cfg.hidden_dropout_prob = hidden_dropout
        cfg.attention_probs_dropout_prob = attention_dropout
        self.bert = AutoModel.from_pretrained(model_name, config=cfg)
        if gradient_checkpointing and hasattr(self.bert, "gradient_checkpointing_enable"):
            self.bert.gradient_checkpointing_enable()
        h = self.bert.config.hidden_size
        self.pre_norm = nn.LayerNorm(h * 2)
        self.classifier = nn.Sequential(
            nn.Dropout(head_dropout),
            nn.Linear(h * 2, h),
            nn.GELU(),
            nn.LayerNorm(h),
            nn.Dropout(head_dropout),
            nn.Linear(h, num_labels),
        )
        cw = class_weights.clone().float() if class_weights is not None else None
        self.loss_fn = WeightedLabelSmoothingCE(weight=cw, smoothing=label_smoothing)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        model_inputs = {"input_ids": input_ids, "attention_mask": attention_mask, "return_dict": True}
        if token_type_ids is not None:
            model_inputs["token_type_ids"] = token_type_ids
        out = self.bert(**model_inputs)
        hidden = out.last_hidden_state
        cls = hidden[:, 0]
        if attention_mask is None:
            mask = torch.ones_like(hidden[:, :, :1])
        else:
            mask = attention_mask.unsqueeze(-1).float()
        mean_pool = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1e-8)
        pooled = self.pre_norm(torch.cat([cls, mean_pool], dim=-1))
        logits = self.classifier(pooled)
        loss = self.loss_fn(logits, labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits, hidden_states=out.hidden_states, attentions=out.attentions)

model = BanglaBertBestHead(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    class_weights=class_weights,
    label_smoothing=LABEL_SMOOTHING,
    hidden_dropout=HIDDEN_DROPOUT,
    attention_dropout=ATTENTION_DROPOUT,
    head_dropout=HEAD_DROPOUT,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Trainable parameters: 111213314


In [8]:
def softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def eval_from_logits(labels, logits):
    labels = np.asarray(labels)
    preds = np.asarray(logits).argmax(-1)
    out = {
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(labels, preds, average="weighted", zero_division=0)),
        "binary_f1": float(f1_score(labels, preds, average="binary", pos_label=1, zero_division=0)),
        "precision": float(precision_score(labels, preds, average="binary", pos_label=1, zero_division=0)),
        "recall": float(recall_score(labels, preds, average="binary", pos_label=1, zero_division=0)),
    }
    return out, preds

def make_compute_metrics():
    def _cm(eval_pred):
        logits, labels = eval_pred
        preds = np.asarray(logits).argmax(-1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
            "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        }
    return _cm

def save_predictions(path, texts, labels, logits, extra=None):
    logits = np.asarray(logits)
    probs = softmax_np(logits)
    preds = logits.argmax(-1)
    df = pd.DataFrame({
        "text": [str(t) for t in texts],
        "gold_label": np.asarray(labels),
        "pred_label": preds,
        "correct": (np.asarray(labels) == preds).astype(int),
        "logit_0": logits[:, 0],
        "logit_1": logits[:, 1],
        "prob_0": probs[:, 0],
        "prob_1": probs[:, 1],
    })
    if extra:
        for k, v in extra.items():
            df[k] = v
    df.to_csv(path, index=False)
    return df

class FGM:
    def __init__(self, model, epsilon=0.5, emb_name="word_embeddings"):
        self.model = model
        self.epsilon = float(epsilon)
        self.emb_name = emb_name
        self.backup = {}
    def attack(self):
        for name, p in self.model.named_parameters():
            if p.requires_grad and self.emb_name in name and p.grad is not None:
                self.backup[name] = p.data.clone()
                grad_norm = torch.norm(p.grad)
                if grad_norm != 0 and not torch.isnan(grad_norm):
                    p.data.add_(self.epsilon * p.grad / grad_norm)
    def restore(self):
        for name, p in self.model.named_parameters():
            if name in self.backup:
                p.data = self.backup[name]
        self.backup = {}

class AWP:
    def __init__(self, model, alpha=0.01, eps=1e-3, param_filter="weight"):
        self.model = model
        self.alpha = float(alpha)
        self.eps = float(eps)
        self.param_filter = param_filter
        self.backup = {}
    def attack(self):
        tiny = 1e-8
        for name, p in self.model.named_parameters():
            if not (p.requires_grad and p.grad is not None and self.param_filter in name):
                continue
            if "LayerNorm" in name or "layer_norm" in name or "bias" in name:
                continue
            self.backup[name] = p.data.clone()
            grad_norm = torch.norm(p.grad)
            weight_norm = torch.norm(p.data.detach())
            if grad_norm != 0 and not torch.isnan(grad_norm):
                r_at = self.alpha * p.grad / (grad_norm + tiny) * (weight_norm + tiny)
                p.data.add_(r_at)
                radius = self.eps * self.backup[name].abs().clamp_min(tiny)
                p.data = torch.max(torch.min(p.data, self.backup[name] + radius), self.backup[name] - radius)
    def restore(self):
        for name, p in self.model.named_parameters():
            if name in self.backup:
                p.data = self.backup[name]
        self.backup = {}

def llrd_param_groups(model, base_lr, head_lr, decay=0.95, weight_decay=0.01):
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    groups = []
    seen = set()

    def add_params(named_params, lr):
        decay_params, nodecay_params = [], []
        for name, p in named_params:
            if not p.requires_grad:
                continue
            (nodecay_params if any(nd in name for nd in no_decay) else decay_params).append(p)
            seen.add(id(p))
        if decay_params:
            groups.append({"params": decay_params, "lr": lr, "weight_decay": weight_decay})
        if nodecay_params:
            groups.append({"params": nodecay_params, "lr": lr, "weight_decay": 0.0})

    bert = model.bert
    layers = list(bert.encoder.layer)
    n_layers = len(layers)
    add_params(list(bert.embeddings.named_parameters()), base_lr * (decay ** (n_layers + 1)))
    for i, layer in enumerate(layers):
        add_params(list(layer.named_parameters()), base_lr * (decay ** (n_layers - i)))
    head_params = [(n, p) for n, p in model.named_parameters() if id(p) not in seen]
    add_params(head_params, head_lr)
    return groups

class FgmAwpTrainer(Trainer):
    def __init__(self, *args, fgm_epsilon=0.5, awp_alpha=0.01, awp_eps=1e-3, awp_start_epoch=1.0,
                 fgm_loss_weight=1.0, awp_loss_weight=1.0,
                 base_lr=2e-5, head_lr=1e-4, llrd_decay=0.95, weight_decay=0.01, **kwargs):
        super().__init__(*args, **kwargs)
        self.fgm = FGM(self.model, epsilon=fgm_epsilon)
        self.awp = AWP(self.model, alpha=awp_alpha, eps=awp_eps)
        self.awp_start_epoch = float(awp_start_epoch)
        self.fgm_loss_weight = float(fgm_loss_weight)
        self.awp_loss_weight = float(awp_loss_weight)
        self.base_lr = base_lr
        self.head_lr = head_lr
        self.llrd_decay = llrd_decay
        self._weight_decay = weight_decay

    def create_optimizer(self):
        if self.optimizer is None:
            groups = llrd_param_groups(self.model, self.base_lr, self.head_lr, self.llrd_decay, self._weight_decay)
            try:
                self.optimizer = torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8, fused=HAS_CUDA)
            except TypeError:
                self.optimizer = torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8)
        return self.optimizer

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)

        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)
        if self.args.n_gpu > 1:
            loss = loss.mean()
        self.accelerator.backward(loss)

        self.fgm.attack()
        with self.compute_loss_context_manager():
            loss_fgm = self.compute_loss(model, inputs) * self.fgm_loss_weight
        if self.args.n_gpu > 1:
            loss_fgm = loss_fgm.mean()
        self.accelerator.backward(loss_fgm)
        self.fgm.restore()

        current_epoch = float(self.state.epoch or 0.0)
        if current_epoch >= self.awp_start_epoch:
            self.awp.attack()
            with self.compute_loss_context_manager():
                loss_awp = self.compute_loss(model, inputs) * self.awp_loss_weight
            if self.args.n_gpu > 1:
                loss_awp = loss_awp.mean()
            self.accelerator.backward(loss_awp)
            self.awp.restore()

        return loss.detach()

In [9]:
run_dir = CKPT / RUN_NAME
if run_dir.exists():
    shutil.rmtree(run_dir, ignore_errors=True)
run_dir.mkdir(parents=True, exist_ok=True)

args_common = dict(
    output_dir=str(run_dir),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,
    max_grad_norm=MAX_GRAD_NORM,
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    fp16=USE_FP16,
    bf16=USE_BF16,
)

try:
    training_args = TrainingArguments(evaluation_strategy="epoch", **args_common)
except TypeError:
    training_args = TrainingArguments(eval_strategy="epoch", **args_common)

trainer = FgmAwpTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=make_compute_metrics(),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    fgm_epsilon=FGM_EPSILON,
    awp_alpha=AWP_ALPHA,
    awp_eps=AWP_EPS,
    awp_start_epoch=AWP_START_EPOCH,
    fgm_loss_weight=FGM_LOSS_WEIGHT,
    awp_loss_weight=AWP_LOSS_WEIGHT,
    base_lr=BASE_LR,
    head_lr=HEAD_LR,
    llrd_decay=LLRD_DECAY,
    weight_decay=WEIGHT_DECAY,
)

print("Training starts...")
t0 = time.time()
train_result = trainer.train()
time_seconds = round(time.time() - t0, 1)
print("Training finished in seconds:", time_seconds)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best eval_loss:", trainer.state.best_metric)

Training starts...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.585000,0.494325,0.779079,0.777778,0.777778
2,0.497300,0.492902,0.789617,0.788208,0.788208
3,0.447400,0.476348,0.803669,0.803432,0.803432
4,0.397800,0.478763,0.798985,0.798593,0.798593
5,0.347100,0.485298,0.795472,0.794412,0.794412


Training finished in seconds: 1084.3
Best checkpoint: /workspace/Sarcasm_detection/04_outputs/test_awp/checkpoints/09b_fgm_awp_best_head/checkpoint-1923
Best eval_loss: 0.4763484597206116


In [10]:
val_logits = trainer.predict(val_ds).predictions
test_logits = trainer.predict(test_ds).predictions

val_m, val_preds = eval_from_logits(val_df[LABEL_COL].values, val_logits)
test_m, test_preds = eval_from_logits(test_df[LABEL_COL].values, test_logits)

print("VAL :", {k: round(v, 4) for k, v in val_m.items()})
print("TEST:", {k: round(v, 4) for k, v in test_m.items()})

save_predictions(
    PRED / f"09b_fgm_awp_{TASK}_val_predictions.csv",
    val_df["text"].values,
    val_df[LABEL_COL].values,
    val_logits,
    extra={"model": RUN_NAME, "task": TASK, "split": "val"},
)
save_predictions(
    PRED / f"09b_fgm_awp_{TASK}_test_predictions.csv",
    test_df["text"].values,
    test_df[LABEL_COL].values,
    test_logits,
    extra={"model": RUN_NAME, "task": TASK, "split": "test"},
)

cm = confusion_matrix(test_df[LABEL_COL].values, test_preds).tolist()
json.dump({"test": cm}, open(TABLES / "09b_fgm_awp_confusion.json", "w"), indent=2)

summary_row = {
    "config": "fgm_awp",
    "method": "fgm_awp_best_head",
    "task": TASK,
    "model": MODEL_NAME,
    "tokenizer": type(tokenizer).__name__,
    "head": "CLS+mean pooling MLP head",
    "max_length": MAX_LENGTH,
    "fgm_epsilon": FGM_EPSILON,
    "awp_alpha": AWP_ALPHA,
    "awp_eps": AWP_EPS,
    "awp_start_epoch": AWP_START_EPOCH,
    "loss_function": f"weighted label-smoothed cross entropy, smoothing={LABEL_SMOOTHING}",
    "optimizer": "AdamW + LLRD + cosine schedule",
    "base_lr": BASE_LR,
    "head_lr": HEAD_LR,
    "llrd_decay": LLRD_DECAY,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "epochs_budget": EPOCHS,
    "patience": PATIENCE,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_eval_loss": trainer.state.best_metric,
    "time_seconds": time_seconds,
    "confusion_matrix": json.dumps(cm),
}
for k, v in val_m.items():
    summary_row[f"val_{k}"] = v
for k, v in test_m.items():
    summary_row[f"test_{k}"] = v

summary = pd.DataFrame([summary_row])
summary.to_csv(TABLES / "09b_fgm_awp_summary.csv", index=False)
summary.to_csv(TABLES / "09b_fgm_awp_multiseed.csv", index=False)
print("Saved summary:", TABLES / "09b_fgm_awp_summary.csv")
display(summary.T)

VAL : {'accuracy': 0.8037, 'macro_f1': 0.8034, 'weighted_f1': 0.8034, 'binary_f1': 0.7966, 'precision': 0.8263, 'recall': 0.7689}
TEST: {'accuracy': 0.8041, 'macro_f1': 0.8038, 'weighted_f1': 0.8038, 'binary_f1': 0.7961, 'precision': 0.8305, 'recall': 0.7644}
Saved summary: /workspace/Sarcasm_detection/04_outputs/test_awp/tables/09b_fgm_awp_summary.csv


,0
config,fgm_awp
method,fgm_awp_best_head
task,ben_sarc_binary
model,csebuetnlp/banglabert
tokenizer,ElectraTokenizerFast
head,CLS+mean pooling MLP head
max_length,192
fgm_epsilon,0.5
awp_alpha,0.01
awp_eps,0.001


In [11]:
history = pd.DataFrame(trainer.state.log_history)
loss_rows = []
for _, r in history.iterrows():
    epoch = r.get("epoch", np.nan)
    if pd.isna(epoch):
        continue
    item = {"epoch": epoch}
    if "loss" in r and not pd.isna(r["loss"]):
        item["train_loss"] = r["loss"]
    if "eval_loss" in r and not pd.isna(r["eval_loss"]):
        item["val_loss"] = r["eval_loss"]
    if "eval_macro_f1" in r and not pd.isna(r["eval_macro_f1"]):
        item["val_macro_f1"] = r["eval_macro_f1"]
    if "eval_accuracy" in r and not pd.isna(r["eval_accuracy"]):
        item["val_accuracy"] = r["eval_accuracy"]
    if len(item) > 1:
        loss_rows.append(item)

curve = pd.DataFrame(loss_rows)
train_curve = curve.dropna(subset=["train_loss"])[["epoch", "train_loss"]] if "train_loss" in curve.columns else pd.DataFrame(columns=["epoch", "train_loss"])
val_curve = curve.dropna(subset=["val_loss"])[["epoch", "val_loss"]] if "val_loss" in curve.columns else pd.DataFrame(columns=["epoch", "val_loss"])
f1_curve = curve.dropna(subset=["val_macro_f1"])[["epoch", "val_macro_f1"]] if "val_macro_f1" in curve.columns else pd.DataFrame(columns=["epoch", "val_macro_f1"])

merged = train_curve.merge(val_curve, on="epoch", how="outer").merge(f1_curve, on="epoch", how="outer").sort_values("epoch")
merged.to_csv(TABLES / "09b_fgm_awp_loss_curve.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
if "train_loss" in merged.columns:
    axes[0].plot(merged["epoch"], merged["train_loss"], "o-", label="Train loss")
if "val_loss" in merged.columns:
    axes[0].plot(merged["epoch"], merged["val_loss"], "s-", label="Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Train vs validation loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

if "val_macro_f1" in merged.columns:
    axes[1].plot(merged["epoch"], merged["val_macro_f1"], "^-", label="Val macro-F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("macro-F1")
axes[1].set_title("Validation macro-F1")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle("NB09b FGM+AWP best-head BanglaBERT — Ben-Sarc only")
fig.tight_layout()
out_fig = FIGS / "09b_fgm_awp_loss_curve.png"
fig.savefig(out_fig, dpi=180, bbox_inches="tight")
plt.show()
print("Saved figure:", out_fig)
display(merged)

Saved figure: /workspace/Sarcasm_detection/04_outputs/test_awp/figures/09b_fgm_awp_loss_curve.png


,epoch,train_loss,val_loss,val_macro_f1
0,1.0,0.5850,0.494325,0.777778
1,2.0,0.4973,0.492902,0.788208
2,3.0,0.4474,0.476348,0.803432
3,4.0,0.3978,0.478763,0.798593
4,5.0,0.3471,0.485298,0.794412


In [12]:
best_dir = CKPT / RUN_NAME / "best_model"
best_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), best_dir / "pytorch_model_state_dict.pt")
tokenizer.save_pretrained(best_dir)
json.dump(config, open(best_dir / "09b_fgm_awp_config.json", "w"), indent=2)
print("Saved checkpoint artifacts:", best_dir)

ensemble_path = ROOT / "04_outputs" / "tables" / "11_ensemble_results.csv"
if ensemble_path.exists():
    ens = pd.read_csv(ensemble_path)
    ens_best = float(ens["test_macro_f1"].max())
    my_f1 = float(test_m["macro_f1"])
    print(f"Best NB11 ensemble macro-F1: {ens_best:.4f}")
    print(f"NB09b FGM+AWP macro-F1    : {my_f1:.4f}")
    print("CROSSES ENSEMBLE" if my_f1 > ens_best else "Does not cross ensemble")
else:
    print("NB11 ensemble file not found; skipping comparison.")

gc.collect()
if HAS_CUDA:
    torch.cuda.empty_cache()

Saved checkpoint artifacts: /workspace/Sarcasm_detection/04_outputs/test_awp/checkpoints/09b_fgm_awp_best_head/best_model
Best NB11 ensemble macro-F1: 0.8108
NB09b FGM+AWP macro-F1    : 0.8038
Does not cross ensemble
